# Data Upload & Insertion — Zeta HOL Sample

This notebook demonstrates how to load the sample metric data JSON file into a pandas DataFrame and perform basic analysis.

**Prerequisites:** Python 3.12+, pandas installed in your session runtime.

## 1. Setup and Imports

In [ ]:
import json
from pathlib import Path

import pandas as pd

## 2. Load Sample Metric Data

Upload `sample_metric_data.json` to your project (via the **Data** tab or by cloning this repo), then update the path below if needed.

In [ ]:
# Adjust this path to where you placed the JSON file in your project
DATA_PATH = Path("sample_metric_data.json")

# Fallback: look in artifacts folder if cloned from zeta-hol repo
if not DATA_PATH.exists():
    DATA_PATH = Path("artifacts/sample_metric_data.json")

with open(DATA_PATH, "r") as f:
    payload = json.load(f)

print(f"Dataset: {payload['dataset']}")
print(f"Version: {payload['version']}")
print(f"Records: {len(payload['records'])}")

## 3. Create DataFrame

In [ ]:
df = pd.DataFrame(payload["records"])
df["date"] = pd.to_datetime(df["date"])

df.head()

## 4. Compute Derived Metrics

In [ ]:
df["ctr"] = (df["clicks"] / df["impressions"] * 100).round(4)
df["conversion_rate"] = (df["conversions"] / df["clicks"] * 100).round(4)
df["roas"] = (df["revenue_usd"] / df["spend_usd"]).round(2)
df["cpc"] = (df["spend_usd"] / df["clicks"]).round(4)

df[["campaign_name", "channel", "ctr", "conversion_rate", "roas", "cpc"]]

## 5. Aggregate by Channel

In [ ]:
channel_summary = (
    df.groupby("channel")
    .agg(
        total_impressions=("impressions", "sum"),
        total_clicks=("clicks", "sum"),
        total_conversions=("conversions", "sum"),
        total_spend=("spend_usd", "sum"),
        total_revenue=("revenue_usd", "sum"),
    )
    .assign(roas=lambda x: (x["total_revenue"] / x["total_spend"]).round(2))
    .sort_values("total_revenue", ascending=False)
)

channel_summary

## 6. Insert Data (Example: Save to CSV / Parquet)

Use these cells to persist processed data for downstream hackathon tasks.

In [ ]:
OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True)

df.to_csv(OUTPUT_DIR / "campaign_metrics_enriched.csv", index=False)
df.to_parquet(OUTPUT_DIR / "campaign_metrics_enriched.parquet", index=False)
channel_summary.to_csv(OUTPUT_DIR / "channel_summary.csv")

print(f"Saved {len(df)} records to {OUTPUT_DIR}/")

## 7. Optional: Insert into a Database

If your hackathon project uses a database, uncomment and adapt the example below.

In [ ]:
# from sqlalchemy import create_engine
#
# engine = create_engine("sqlite:///output/metrics.db")
# df.to_sql("campaign_metrics", engine, if_exists="replace", index=False)
# channel_summary.to_sql("channel_summary", engine, if_exists="replace")
# print("Data inserted into metrics.db")